# Clasificación de sentimiento con BERT
### Enfoque 2 de 3 — BERT como capa de embeddings congelada

**Universidad de Santiago de Chile (USACH)**  
Diplomado de Inteligencia artificial aplicada a la industria tecnológica  
Autor: Diego Fischer

BERT actúa como **extractor de características congelado** (`trainable=False`). Sus *embeddings* contextuales (`sequence_output`) alimentan una CNN; solo se entrena la CNN. Con KerasHub usamos `keras_hub.models.BertBackbone`.

> Los tres cuadernos de esta serie resuelven la misma tarea (clasificación binaria de sentimiento sobre el dataset Sentiment140) y se diferencian únicamente en el rol que cumple BERT dentro del modelo.

> **Migrado a KerasHub.** El paquete `bert-for-tf2` quedó obsoleto y no funciona con Keras 3 (el *default* en Colab actual). Aquí usamos [KerasHub](https://keras.io/keras_hub/), la biblioteca nativa de Keras 3 para modelos preentrenados.

---

# Fase 1: Importar las dependencias

Instalamos **KerasHub**. Requiere Keras 3, que ya viene por defecto en Colab, así que no hace falta reiniciar el entorno.

In [ ]:
!pip install -q -U keras-hub

Importamos las librerías. Fijamos TensorFlow como *backend* de Keras 3 **antes** de importar `keras`. `keras_hub` nos da el tokenizador y el modelo BERT preentrenados.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # backend de Keras 3

import numpy as np
import math
import re
import pandas as pd
from bs4 import BeautifulSoup
import random

import tensorflow as tf
import keras
import keras_hub
from tensorflow.keras import layers

from google.colab import drive

# Preset de BERT preentrenado (equivalente a bert_en_uncased_L-12_H-768_A-12)
BERT_PRESET = "bert_base_en_uncased"

Montamos Google Drive para leer el dataset.

In [ ]:
drive.mount("/content/drive")

# Fase 2: Preprocesamiento de datos

## Carga de los ficheros

Leemos el CSV de entrenamiento de Sentiment140 (seis columnas; `encoding='latin1'` por los caracteres no UTF-8 de los tweets).

In [ ]:
cols = ["sentiment", "id", "date", "query", "user", "text"]
data = pd.read_csv(
    "/content/drive/My Drive/Curso de NLP/BERT/sentiment_data/training.csv",
    header=None,
    names=cols,
    engine="python",
    encoding="latin1"
)

Conservamos solo las columnas útiles: `sentiment` y `text`.

In [ ]:
data.drop(["id", "date", "query", "user"], axis=1, inplace=True)

## Limpieza

Función que limpia cada tweet: quita el HTML, las menciones `@usuario`, las URLs y todo lo que no sean letras o signos básicos, y colapsa los espacios sobrantes.

In [ ]:
def clean_tweet(tweet):
    tweet = BeautifulSoup(tweet, "lxml").get_text()
    # Eliminar el @ y su mención
    tweet = re.sub(r"@[A-Za-z0-9]+", ' ', tweet)
    # Eliminar los links de la URL
    tweet = re.sub(r"https?://[A-Za-z0-9./]+", ' ', tweet)
    # Conservar solo las letras y algunos signos
    tweet = re.sub(r"[^a-zA-Z.!?']", ' ', tweet)
    # Eliminar espacios en blanco adicionales
    tweet = re.sub(r" +", ' ', tweet)
    return tweet

Aplicamos la limpieza a todos los tweets.

In [ ]:
data_clean = [clean_tweet(tweet) for tweet in data.text]

Extraemos las etiquetas. En Sentiment140 el positivo viene como `4`; lo pasamos a `1` para tener etiquetas binarias **0 (negativo)** / **1 (positivo)**.

In [ ]:
data_labels = data.sentiment.values
data_labels[data_labels == 4] = 1

## Tokenización y empaquetado

Usamos el **preprocesador de BERT** de KerasHub: tokeniza, agrega `[CLS]`/`[SEP]` y arma el diccionario `{token_ids, segment_ids, padding_mask}` que espera el modelo, todo a una longitud fija de secuencia.

In [ ]:
SEQ_LEN = 40   # los tweets son cortos; longitud fija de secuencia
preprocessor = keras_hub.models.BertTextClassifierPreprocessor.from_preset(
    BERT_PRESET, sequence_length=SEQ_LEN)

## Creación del dataset

Armamos un `tf.data.Dataset`, lo mezclamos, lo separamos en entrenamiento/prueba y aplicamos el preprocesador por lotes (`map`).

In [ ]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

full_ds = tf.data.Dataset.from_tensor_slices((data_clean, data_labels))
full_ds = full_ds.shuffle(len(data_clean), seed=42)

NB_TEST = len(data_clean) // 10

def make_ds(ds):
    return (ds.batch(BATCH_SIZE)
              .map(lambda x, y: (preprocessor(x), y), num_parallel_calls=AUTOTUNE)
              .prefetch(AUTOTUNE))

test_dataset  = make_ds(full_ds.take(NB_TEST))
train_dataset = make_ds(full_ds.skip(NB_TEST))

> Nota: pasar BERT (aunque esté congelado) sobre 1,6 M de tweets es costoso. Para probar rápido en clase puedes usar un subconjunto, p. ej. `train_dataset = train_dataset.take(500)`.

# Fase 3: Construcción del modelo

El modelo toma el `sequence_output` de BERT (los *embeddings* contextuales, uno por token) y lo pasa por la misma CNN de bi/tri/tetragramas. **BERT queda congelado** (`self.bert.trainable = False`): solo se entrenan las convoluciones y las densas.

In [ ]:
class DCNNBERTEmbedding(tf.keras.Model):
    def __init__(self, bert_preset, nb_filters=50, FFN_units=512,
                 nb_classes=2, dropout_rate=0.1, name="dcnn_bert"):
        super().__init__(name=name)
        self.bert = keras_hub.models.BertBackbone.from_preset(bert_preset)
        self.bert.trainable = False   # BERT congelado: solo extractor de características
        self.bigram   = layers.Conv1D(nb_filters, 2, padding="valid", activation="relu")
        self.trigram  = layers.Conv1D(nb_filters, 3, padding="valid", activation="relu")
        self.fourgram = layers.Conv1D(nb_filters, 4, padding="valid", activation="relu")
        self.pool = layers.GlobalMaxPool1D()
        self.dense_1 = layers.Dense(units=FFN_units, activation="relu")
        self.dropout = layers.Dropout(rate=dropout_rate)
        if nb_classes == 2:
            self.last_dense = layers.Dense(units=1, activation="sigmoid")
        else:
            self.last_dense = layers.Dense(units=nb_classes, activation="softmax")

    def call(self, inputs, training=False):
        x = self.bert(inputs)["sequence_output"]   # (batch, seq_len, 768)
        x_1 = self.pool(self.bigram(x))
        x_2 = self.pool(self.trigram(x))
        x_3 = self.pool(self.fourgram(x))
        merged = tf.concat([x_1, x_2, x_3], axis=-1)
        merged = self.dense_1(merged)
        merged = self.dropout(merged, training=training)
        return self.last_dense(merged)

# Fase 4: Entrenamiento

Instanciamos y compilamos.

In [ ]:
NB_FILTERS = 100
FFN_UNITS = 256
NB_CLASSES = 2
DROPOUT_RATE = 0.2
NB_EPOCHS = 5

Dcnn = DCNNBERTEmbedding(BERT_PRESET, nb_filters=NB_FILTERS, FFN_units=FFN_UNITS,
                         nb_classes=NB_CLASSES, dropout_rate=DROPOUT_RATE)

Dcnn.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

Guardamos los pesos en Drive por época.

In [ ]:
checkpoint_path = "/content/drive/My Drive/Curso de NLP/BERT/ckpt_bert_embedding.weights.h5"
ckpt_cb = keras.callbacks.ModelCheckpoint(checkpoint_path, save_weights_only=True)

Entrenamos (solo se ajustan la CNN y las densas; BERT no cambia).

In [ ]:
Dcnn.fit(train_dataset, epochs=NB_EPOCHS, callbacks=[ckpt_cb])

# Fase 5: Evaluación

In [ ]:
results = Dcnn.evaluate(test_dataset)
print(results)

Probamos con una frase nueva. Reusamos el preprocesador para armar la entrada.

In [ ]:
def get_prediction(sentence):
    inputs = preprocessor([sentence])   # dict con un lote de tamaño 1
    output = Dcnn(inputs, training=False)
    sentiment = math.floor(float(output) * 2)
    etiqueta = "Positivo" if sentiment == 1 else "Negativo"
    print("Salida del modelo: {}\nSentimiento predicho: {}.".format(float(output), etiqueta))

get_prediction("This actor is a deception.")

---
## Licencia

[![Licencia Creative Commons](https://i.creativecommons.org/l/by/4.0/88x31.png)](https://creativecommons.org/licenses/by/4.0/deed.es)

Esta obra está licenciada bajo una [Licencia Creative Commons Atribución 4.0 Internacional (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/deed.es).

**Atribución:** Diego Fischer — Universidad de Santiago de Chile (USACH).